# 分词器与模型配合使用

In [2]:
import torch
from torchgen.api.cpp import return_type

# 准备数据
texts = ["我爱自然语言处理", "你爱打篮球", "我们一起成长"]

In [34]:
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification

# 加载模型和分词器
model_name = "google-bert/bert-base-chinese"
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [7]:
# 调用 tokenizer
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)
print(inputs)

{'input_ids': tensor([[ 101, 2769, 4263, 5632, 4197, 6427, 6241, 1905, 4415,  102],
        [ 101,  872, 4263, 2802, 5074, 4413,  102,    0,    0,    0],
        [ 101, 2769,  812,  671, 6629, 2768, 7270,  102,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]])}


In [12]:
# 将 inputs 中的每个元素，作为参数传入，前向传播推理
import torch

with torch.no_grad():
    outputs = model(**inputs)
print(outputs)

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-3.1871e-01, -1.4260e-01, -2.1351e-01,  ...,  2.9723e-01,
           2.2833e-01, -2.9379e-01],
         [ 6.7323e-01,  1.2139e-01,  3.0185e-01,  ..., -1.1036e+00,
          -4.9975e-02,  8.7921e-02],
         [ 3.6870e-01, -5.3108e-01, -6.8611e-01,  ...,  1.9168e-01,
          -1.7676e-02, -2.7852e-01],
         ...,
         [-4.4937e-04, -1.3693e-01,  1.2886e+00,  ..., -6.4702e-02,
          -3.4146e-01, -1.6366e-01],
         [ 6.7370e-01,  7.9813e-01,  6.0918e-02,  ..., -3.2120e-01,
           2.8577e-01, -1.1626e-01],
         [ 3.5677e-01, -3.3869e-01,  1.3571e+00,  ...,  4.6651e-01,
          -7.7459e-03,  1.9304e-01]],

        [[-4.7717e-02,  2.1300e-01, -1.3359e-02,  ...,  1.0403e+00,
           3.0241e-01, -2.7685e-01],
         [-8.2625e-02, -1.1088e-01, -4.8450e-01,  ..., -7.5406e-01,
          -4.1614e-01, -1.5695e-01],
         [-5.6740e-02,  2.8971e-01, -1.2939e+00,  ...,  1.6063e-02,
           4.

In [13]:
print(outputs.last_hidden_state.shape)

torch.Size([3, 10, 768])


In [14]:
print(outputs.pooler_output.shape)

torch.Size([3, 768])


# 带任务头的模型

In [18]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

In [21]:
with torch.no_grad():
    outputs = model(**inputs)
    print(outputs)

SequenceClassifierOutput(loss=None, logits=tensor([[ 0.8486,  0.0093],
        [ 0.5042, -0.0978],
        [ 0.5525, -0.2199]]), hidden_states=None, attentions=None)


# Dataset

In [2]:
from datasets import load_dataset, load_from_disk

# 加载文件
dataset_dict = load_dataset(
    path='json',
    data_files={
        'train': './train.jsonl',
        'test': './test.jsonl'
    }
)

/opt/miniconda3/envs/python3.14/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['en', 'zh'],
        num_rows: 23324
    })
    test: Dataset({
        features: ['en', 'zh'],
        num_rows: 5831
    })
})


# 数据预处理

In [11]:
dataset_dict = load_dataset(
    'csv',
    data_files='./online_shopping_10_cats.csv'
)

In [7]:
print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['cat', 'label', 'review'],
        num_rows: 62774
    })
})


In [24]:
dataset = dataset_dict['train']

In [25]:
print(dataset)

Dataset({
    features: ['cat', 'label', 'review'],
    num_rows: 62774
})


# 删除列

In [26]:
dataset = dataset.remove_columns(['cat'])
print(dataset)

Dataset({
    features: ['label', 'review'],
    num_rows: 62774
})


# 删除列

In [27]:
dataset = dataset.filter(lambda x: x['review'] is not None)
print(dataset)

Filter: 100%|██████████| 62774/62774 [00:00<00:00, 904613.68 examples/s]

Dataset({
    features: ['label', 'review'],
    num_rows: 62773
})


# 划分数据集

In [29]:
dataset_dict = dataset.train_test_split(test_size=0.2)
print(dataset_dict)

Parameter 'generator'=Generator(PCG64) of the transform datasets.arrow_dataset.Dataset.train_test_split couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 50218
    })
    test: Dataset({
        features: ['label', 'review'],
        num_rows: 12555
    })
})


# 数据编码

In [65]:
dataset = dataset_dict['train']
print(dataset)

Dataset({
    features: ['label', 'review'],
    num_rows: 50218
})


In [66]:
dataset = dataset.map(lambda x: tokenizer(x['review'], truncation=True), batched=True)
print(dataset)

Dataset({
    features: ['label', 'review', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 50218
})


In [67]:
dataset = dataset.map(lambda x: tokenizer(x['review'], truncation=True), batched=True, remove_columns=['review'])
print(dataset)

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 50218
})


In [68]:
dataset = dataset.rename_columns({'label': 'labels'})
print(dataset)

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 50218
})


In [69]:
dataset_dict['train'] = dataset
print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 50218
    })
    test: Dataset({
        features: ['label', 'review'],
        num_rows: 12555
    })
})


# 数据保存

In [70]:
dataset_dict.save_to_disk('./data/processed')

Saving the dataset (1/1 shards): 100%|██████████| 12555/12555 [00:00<00:00, 667429.08 examples/s]


# 加载数据

In [72]:
from datasets import load_from_disk

dataset_dict = load_from_disk('../data/processed')
print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 50218
    })
    test: Dataset({
        features: ['label', 'review'],
        num_rows: 12555
    })
})


# to json

In [73]:
dataset_dict['train'].to_json('./data/train.jsonl')

Creating json from Arrow format: 100%|██████████| 51/51 [00:00<00:00, 506.29ba/s]


18043609

# 加载 json 文件

In [77]:
train_dataset_dict = load_dataset("json", data_files="./data/train.jsonl")
print(train_dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 50218
    })
})


# to csv

In [78]:
dataset_dict['train'].to_csv('./data/train.csv')

Creating CSV from Arrow format: 100%|██████████| 51/51 [00:00<00:00, 709.90ba/s]


8651176

# 加载 csv

In [80]:
train_dataset_dict = load_dataset("csv", data_files="./data/train.csv")
print(train_dataset_dict)

Extracting data files: 100%|██████████| 1/1 [00:00<00:00, 465.98it/s]
Generating train split: 0 examples [00:00, ? examples/s]/opt/miniconda3/envs/python3.14/lib/python3.13/site-packages/datasets/download/streaming_download_manager.py:765: FutureWarning: The 'verbose' keyword in pd.read_csv is deprecated and will be removed in a future version.
  return pd.read_csv(xopen(filepath_or_buffer, "rb", download_config=download_config), **kwargs)
Generating train split: 50218 examples [00:00, 478393.67 examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 50218
    })
})


# 集成 dataloader

In [142]:
dataset_dict = load_dataset(
    path='csv',
    data_files={'train': './online_shopping_10_cats.csv'}
)
print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['cat', 'label', 'review'],
        num_rows: 62774
    })
})


In [143]:
# 删除列
dataset_dict = dataset_dict.remove_columns(['cat'])
print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 62774
    })
})


In [144]:
# 过滤数据
dataset_dict = dataset_dict.filter(lambda x: x['review'] is not None)
print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 62773
    })
})


In [145]:
# 划分数据集
train_dataset = dataset_dict['train']
print(train_dataset)
dataset_dict = train_dataset.train_test_split(test_size=0.2)
print(dataset_dict)

Dataset({
    features: ['label', 'review'],
    num_rows: 62773
})
DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 50218
    })
    test: Dataset({
        features: ['label', 'review'],
        num_rows: 12555
    })
})


In [146]:
# 编码
dataset_dict = dataset_dict.map(lambda x: tokenizer(x['review'], truncation=True, padding=True, return_tensors="pt"),
                                batched=True,
                                remove_columns=['review'])
dataset_dict = dataset_dict.rename_columns({'label': 'labels'})
print(dataset_dict)



Map: 100%|██████████| 12555/12555 [00:03<00:00, 3828.79 examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50218
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 12555
    })
})


In [147]:
# 保存数据集
dataset_dict.save_to_disk('./data/processed')

Saving the dataset (1/1 shards): 100%|██████████| 12555/12555 [00:00<00:00, 814519.29 examples/s]


In [148]:
# 读取数据集
train_dataset = load_from_disk('../data/processed/train')
print(train_dataset)

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 50218
})


In [149]:

from torch.utils.data import DataLoader

dataloader = DataLoader(train_dataset, batch_size=16)
data_iter = iter(dataloader)
batch = next(data_iter)
print(batch)

{'labels': tensor([0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0]), 'input_ids': [tensor([101, 101, 101, 101, 101, 101, 101, 101, 101, 101, 101, 101, 101, 101,
        101, 101]), tensor([6843, 2376, 4959,  741, 6821, 2140,  679, 9048, 6983, 2991, 5439, 6820,
        6821, 2376, 2902,  776]), tensor([4638, 3301,  749, 4638, 3613, 6564, 7231,  743, 2421, 2429, 1062, 1762,
        3221, 2769, 7241,  691]), tensor([1259, 1351,  676, 1079, 3221, 6574, 4802, 4638,  769, 1372, 4500,  886,
        5018, 6163,  679, 4289]), tensor([1259, 2157, 3613, 2159, 1346, 7030, 2141, 4007, 6858, 3300, 6814, 4500,
         753, 5143, 3221, 3837]), tensor([1922, 4638, 3123, 1157, 1217,  679,  679, 4007,  679,  671,  671, 2496,
        3613, 5320, 2523, 1282]), tensor([1920, 2111, 3819, 4692, 3855, 7231, 7231,  671, 3221,  702, 4486,  704,
         743, 4638, 5653, 1146]), tensor([ 749, 2094, 6132, 1962,  691, 8024, 6574, 1920, 2523,  511,  749, 7509,
        8024, 2360, 3302,  679]), tensor([8024,  743, 3